In [2]:
import os
import sys
#print(sys.executable)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import root_scalar
from scipy.integrate import quad
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.integrate import trapezoid
import glob

In [21]:
ref_87 = 384230484468500
P_3_87 = 193740700
P_2_87 = -72911200
P_1_87 = -229851800
P_0_87 = -302073800
S_2_87 = 2563005979.089109
S_1_87 = -4271676631.815181

ref_85 = 384230406373000
P_4_85 = 100357000
P_3_85 = -20503000
P_2_85 = -83955000
P_1_85 = -113307000
S_3_85 = 1264888516.3
S_2_85 = -1770843922.8

# -----------------------------
# Hyperfine constants (MHz)
# -----------------------------
# Rb-87
P_87 = {0: P_0_87, 1: P_1_87, 2: P_2_87, 3: P_3_87}
S_87 = {1: S_1_87, 2: S_2_87}

# Rb-85
P_85 = {1: P_1_85, 2: P_2_85, 3: P_3_85, 4: P_4_85}
S_85 = {2: S_2_85, 3: S_3_85}

# -----------------------------
# 1) Allowed transitions
# -----------------------------
transitions_87 = {}
transitions_85 = {}

# Rb-87: F=1,2 → F'=0,1,2,3
for F in [1, 2]:
    for Fp in [0, 1, 2, 3]:
        if abs(F - Fp) <= 1:
            freq = ref_87 + P_87[Fp] - S_87[F]
            transitions_87[(F, Fp)] = freq

# Rb-85: F=2,3 → F'=1,2,3,4
for F in [2, 3]:
    for Fp in [1, 2, 3, 4]:
        if abs(F - Fp) <= 1:
            freq = ref_85 + P_85[Fp] - S_85[F]
            transitions_85[(F, Fp)] = freq

# -----------------------------
# 2) Crossover transitions
# -----------------------------
cross_87 = {}
cross_85 = {}

# 87 crossovers
keys = list(transitions_87.keys())
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        (F1, Fp1) = keys[i]
        (F2, Fp2) = keys[j]
        if F1 == F2:
            f1 = transitions_87[(F1, Fp1)]
            f2 = transitions_87[(F2, Fp2)]
            cross_87[((F1, Fp1), (F2, Fp2))] = 0.5*(f1 + f2)

# 85 crossovers
keys = list(transitions_85.keys())
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        (F1, Fp1) = keys[i]
        (F2, Fp2) = keys[j]
        if F1 == F2:
            f1 = transitions_85[(F1, Fp1)]
            f2 = transitions_85[(F2, Fp2)]
            cross_85[((F1, Fp1), (F2, Fp2))] = 0.5*(f1 + f2)

# -----------------------------
# 3) Print results
# -----------------------------
print("=== Rb-87 Allowed transitions (THz) ===")
for k, v in transitions_87.items():
    print(f"F={k[0]} → F'={k[1]} : {v/1e12} THz")

print("\n=== Rb-87 Crossovers (THz) ===")
for k, v in cross_87.items():
    (F1, Fp1), (F2, Fp2) = k
    print(f"(F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2}) : {v/1e12} THz")

print("\n=== Rb-85 Allowed transitions (THz) ===")
for k, v in transitions_85.items():
    print(f"F={k[0]} → F'={k[1]} : {v/1e12} THz")

print("\n=== Rb-85 Crossovers (THz) ===")
for k, v in cross_85.items():
    (F1, Fp1), (F2, Fp2) = k
    print(f"(F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2}) : {v/1e12} THz")


=== Rb-87 Allowed transitions (THz) ===
F=1 → F'=0 : 384.2344540713318 THz
F=1 → F'=1 : 384.23452629333184 THz
F=1 → F'=2 : 384.23468323393183 THz
F=2 → F'=1 : 384.2276916107209 THz
F=2 → F'=2 : 384.22784855132096 THz
F=2 → F'=3 : 384.2281152032209 THz

=== Rb-87 Crossovers (THz) ===
(F=1→F'=0) × (F=1→F'=1) : 384.2344901823318 THz
(F=1→F'=0) × (F=1→F'=2) : 384.2345686526318 THz
(F=1→F'=1) × (F=1→F'=2) : 384.2346047636318 THz
(F=2→F'=1) × (F=2→F'=2) : 384.22777008102094 THz
(F=2→F'=1) × (F=2→F'=3) : 384.22790340697094 THz
(F=2→F'=2) × (F=2→F'=3) : 384.22798187727096 THz

=== Rb-85 Allowed transitions (THz) ===
F=2 → F'=1 : 384.23206390992283 THz
F=2 → F'=2 : 384.2320932619228 THz
F=2 → F'=3 : 384.2321567139228 THz
F=3 → F'=2 : 384.22905752948367 THz
F=3 → F'=3 : 384.2291209814837 THz
F=3 → F'=4 : 384.2292418414837 THz

=== Rb-85 Crossovers (THz) ===
(F=2→F'=1) × (F=2→F'=2) : 384.2320785859228 THz
(F=2→F'=1) × (F=2→F'=3) : 384.23211031192284 THz
(F=2→F'=2) × (F=2→F'=3) : 384.232124987922

In [22]:
all_lines = []

# -----------------------------
# 1) Allowed transitions (87)
# -----------------------------
for (F, Fp), v in transitions_87.items():
    freq = v
    label = f"Rb87: F={F} → F'={Fp}"
    all_lines.append((freq, label))

# -----------------------------
# 2) Crossover transitions (87)
# -----------------------------
for k, v in cross_87.items():
    (F1, Fp1), (F2, Fp2) = k
    freq = v
    label = f"Rb87: (F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2})"
    all_lines.append((freq, label))

# -----------------------------
# 3) Allowed transitions (85)
# -----------------------------
for (F, Fp), v in transitions_85.items():
    freq = v
    label = f"Rb85: F={F} → F'={Fp}"
    all_lines.append((freq, label))

# -----------------------------
# 4) Crossover transitions (85)
# -----------------------------
for k, v in cross_85.items():
    (F1, Fp1), (F2, Fp2) = k
    freq = v
    label = f"Rb85: (F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2})"
    all_lines.append((freq, label))

# -----------------------------
# 5) Sort by frequency
# -----------------------------
all_lines_sorted = sorted(all_lines, key=lambda x: x[0])

# -----------------------------
# 6) Print
# -----------------------------
print("=== All transitions (Rb-87 + Rb-85, real + crossover) sorted by frequency (THz) ===")
for freq, label in all_lines_sorted:
    print(f"{freq/1e12} THz  :  {label}")

=== All transitions (Rb-87 + Rb-85, real + crossover) sorted by frequency (THz) ===
384.2276916107209 THz  :  Rb87: F=2 → F'=1
384.22777008102094 THz  :  Rb87: (F=2→F'=1) × (F=2→F'=2)
384.22784855132096 THz  :  Rb87: F=2 → F'=2
384.22790340697094 THz  :  Rb87: (F=2→F'=1) × (F=2→F'=3)
384.22798187727096 THz  :  Rb87: (F=2→F'=2) × (F=2→F'=3)
384.2281152032209 THz  :  Rb87: F=2 → F'=3
384.22905752948367 THz  :  Rb85: F=3 → F'=2
384.2290892554837 THz  :  Rb85: (F=3→F'=2) × (F=3→F'=3)
384.2291209814837 THz  :  Rb85: F=3 → F'=3
384.22914968548366 THz  :  Rb85: (F=3→F'=2) × (F=3→F'=4)
384.2291814114837 THz  :  Rb85: (F=3→F'=3) × (F=3→F'=4)
384.2292418414837 THz  :  Rb85: F=3 → F'=4
384.23206390992283 THz  :  Rb85: F=2 → F'=1
384.2320785859228 THz  :  Rb85: (F=2→F'=1) × (F=2→F'=2)
384.2320932619228 THz  :  Rb85: F=2 → F'=2
384.23211031192284 THz  :  Rb85: (F=2→F'=1) × (F=2→F'=3)
384.2321249879228 THz  :  Rb85: (F=2→F'=2) × (F=2→F'=3)
384.2321567139228 THz  :  Rb85: F=2 → F'=3
384.2344540713318

In [23]:
rb87 = []
rb85 = []

for freq, label in all_lines_sorted:
    if label.startswith("Rb87"):
        rb87.append((freq, label))
    elif label.startswith("Rb85"):
        rb85.append((freq, label))
print(rb85)

[(384229057529483.7, "Rb85: F=3 → F'=2"), (384229089255483.7, "Rb85: (F=3→F'=2) × (F=3→F'=3)"), (384229120981483.7, "Rb85: F=3 → F'=3"), (384229149685483.7, "Rb85: (F=3→F'=2) × (F=3→F'=4)"), (384229181411483.7, "Rb85: (F=3→F'=3) × (F=3→F'=4)"), (384229241841483.7, "Rb85: F=3 → F'=4"), (384232063909922.8, "Rb85: F=2 → F'=1"), (384232078585922.8, "Rb85: (F=2→F'=1) × (F=2→F'=2)"), (384232093261922.8, "Rb85: F=2 → F'=2"), (384232110311922.8, "Rb85: (F=2→F'=1) × (F=2→F'=3)"), (384232124987922.8, "Rb85: (F=2→F'=2) × (F=2→F'=3)"), (384232156713922.8, "Rb85: F=2 → F'=3")]


In [24]:

def compute_normalized_intervals(group):
    freqs = np.array([f for f, lab in group])
    labels = [lab for f, lab in group]

    intervals = []  # (norm_df, df, label_i, label_j)

    # 모든 조합 (i < j)
    for i in range(len(freqs)):
        for j in range(i+1, len(freqs)):
            df = freqs[j] - freqs[i]
            intervals.append((df, labels[i], labels[j]))

    # 간격만 추출해서 정규화
    df_values = np.array([x[0] for x in intervals])
    max_df = df_values.max()

    normalized = []
    for df, lab1, lab2 in intervals:
        norm_df = df / max_df
        normalized.append((norm_df, df, lab1, lab2))

    # 정규화된 간격 순으로 정렬
    normalized.sort(key=lambda x: x[0])
    return normalized

norm_intervals_87 = compute_normalized_intervals(rb87)
norm_intervals_85 = compute_normalized_intervals(rb85)
print("=== Rb-87 normalized intervals ===")
for norm_df, df, lab1, lab2 in norm_intervals_87:
    print(f"norm={norm_df:.6f}   df={df/1e12:.9f} THz   :  {lab1} ↔ {lab2}")

print("\n=== Rb-85 normalized intervals ===")
for norm_df, df, lab1, lab2 in norm_intervals_85:
    print(f"norm={norm_df:.6f}   df={df/1e12:.9f} THz   :  {lab1} ↔ {lab2}")

=== Rb-87 normalized intervals ===
norm=0.005165   df=0.000036111 THz   :  Rb87: F=1 → F'=0 ↔ Rb87: (F=1→F'=0) × (F=1→F'=1)
norm=0.005165   df=0.000036111 THz   :  Rb87: (F=1→F'=0) × (F=1→F'=1) ↔ Rb87: F=1 → F'=1
norm=0.005165   df=0.000036111 THz   :  Rb87: (F=1→F'=0) × (F=1→F'=2) ↔ Rb87: (F=1→F'=1) × (F=1→F'=2)
norm=0.006059   df=0.000042359 THz   :  Rb87: F=1 → F'=1 ↔ Rb87: (F=1→F'=0) × (F=1→F'=2)
norm=0.007846   df=0.000054856 THz   :  Rb87: F=2 → F'=2 ↔ Rb87: (F=2→F'=1) × (F=2→F'=3)
norm=0.010330   df=0.000072222 THz   :  Rb87: F=1 → F'=0 ↔ Rb87: F=1 → F'=1
norm=0.011223   df=0.000078470 THz   :  Rb87: F=2 → F'=1 ↔ Rb87: (F=2→F'=1) × (F=2→F'=2)
norm=0.011223   df=0.000078470 THz   :  Rb87: (F=2→F'=1) × (F=2→F'=2) ↔ Rb87: F=2 → F'=2
norm=0.011223   df=0.000078470 THz   :  Rb87: (F=2→F'=1) × (F=2→F'=3) ↔ Rb87: (F=2→F'=2) × (F=2→F'=3)
norm=0.011223   df=0.000078470 THz   :  Rb87: (F=1→F'=0) × (F=1→F'=1) ↔ Rb87: (F=1→F'=0) × (F=1→F'=2)
norm=0.011223   df=0.000078470 THz   :  Rb87: F=1

In [25]:
data4_T = [0.0114427, 0.0120711, 0.012397, 0.0126802, 0.0130316]
data4_1 = [0.2874390050049227, 0.33302702376055976, 0.31881418486575, 0.2970975120096744, 0.3008597511899701]
data4_2 = [2.06891, 2.07812, 2.07953, 2.07188, 2.05031]
data6_T = [0.011453, 0.0120791, 0.0124029, 0.0130008]
data6_1 = [-0.72052778409326, -1.8063276656924325, -1.3524347551805662, -0.5659081251481841 ]
data6_2 = [2.0748070597035704, 2.0749573162771995, 2.076009878367027, 2.075763212476051]


In [26]:
from itertools import combinations

# -----------------------------
# 1) Experimental ratio
# -----------------------------
exp = np.array(data6_T)
exp = np.sort(exp)
exp_d = np.diff(exp)
exp_ratio = exp_d / exp_d[0]   # 3개 비율
print("Experimental ratio:", exp_ratio)

# -----------------------------
# 2) 함수: 4개 전이 조합에서 간격 비율 비교
# -----------------------------
def find_matching_groups(group, exp_ratio, tol=0.05):
    matches = []
    freqs = np.array([f for f, lab in group])
    labels = [lab for f, lab in group]

    for idx in combinations(range(len(freqs)), 4):
        block = freqs[list(idx)]
        block_labels = [labels[i] for i in idx]

        block = np.sort(block)
        d = np.diff(block)

        if d[0] == 0:
            continue

        ratio = d / d[0]
        rel_err = np.abs(ratio - exp_ratio) / exp_ratio

        if np.all(rel_err < tol):
            matches.append((block, block_labels, ratio))

    return matches


matches_87 = find_matching_groups(rb87, exp_ratio)
matches_85 = find_matching_groups(rb85, exp_ratio)
print("=== Matches in Rb-87 ===")
for block, labels, ratio in matches_87:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

print("\n=== Matches in Rb-85 ===")
for block, labels, ratio in matches_85:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

Experimental ratio: [1.         0.51716978 0.95495927]
=== Matches in Rb-87 ===

=== Matches in Rb-85 ===

Matched ratio: [1.         0.52500414 1.        ]
  384.229089 THz : Rb85: (F=3→F'=2) × (F=3→F'=3)
  384.229150 THz : Rb85: (F=3→F'=2) × (F=3→F'=4)
  384.229181 THz : Rb85: (F=3→F'=3) × (F=3→F'=4)
  384.229242 THz : Rb85: F=3 → F'=4


In [27]:

from itertools import combinations

# -----------------------------
# 1) Experimental ratio (generalized)
# -----------------------------
data4_T = [0.0114427, 0.0120711, 0.012397, 0.0130316]
exp = np.array(data4_T)


exp = np.sort(exp)
exp_d = np.diff(exp)              # N-1개 간격
exp_ratio = exp_d / exp_d[0]      # 비율
N = len(exp)                      # 실험 포인트 개수

print("Experimental ratio:", exp_ratio)

# -----------------------------
# 2) Generalized matching function
# -----------------------------
def find_matching_groups(group, exp_ratio, N, tol=0.15):
    matches = []
    freqs = np.array([f for f, lab in group])
    labels = [lab for f, lab in group]

    for idx in combinations(range(len(freqs)), N):
        block = freqs[list(idx)]
        block_labels = [labels[i] for i in idx]

        block = np.sort(block)
        d = np.diff(block)

        if d[0] == 0:
            continue

        ratio = d / d[0]

        # 올바른 상대 오차 계산 (모델 기준)
        rel_err = np.abs(ratio - exp_ratio) / ratio

        # 또는 다음 중 하나 택해도 됨:
        # rel_err = np.abs(ratio - exp_ratio)  # 절대 오차
        # rel_err = np.abs(ratio - exp_ratio) / exp_ratio.mean()

        if np.all(rel_err < tol):
            matches.append((block, block_labels, ratio))

    return matches




matches_87 = find_matching_groups(rb87, exp_ratio, N)
matches_85 = find_matching_groups(rb85, exp_ratio, N)
print("=== Matches in Rb-87 ===")
for block, labels, ratio in matches_87:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

print("\n=== Matches in Rb-85 ===")
for block, labels, ratio in matches_85:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

Experimental ratio: [1.         0.51861871 1.00986633]
=== Matches in Rb-87 ===

Matched ratio: [1.         0.58855984 1.        ]
  384.227770 THz : Rb87: (F=2→F'=1) × (F=2→F'=2)
  384.227903 THz : Rb87: (F=2→F'=1) × (F=2→F'=3)
  384.227982 THz : Rb87: (F=2→F'=2) × (F=2→F'=3)
  384.228115 THz : Rb87: F=2 → F'=3

Matched ratio: [1.         0.46018685 1.        ]
  384.234490 THz : Rb87: (F=1→F'=0) × (F=1→F'=1)
  384.234569 THz : Rb87: (F=1→F'=0) × (F=1→F'=2)
  384.234605 THz : Rb87: (F=1→F'=1) × (F=1→F'=2)
  384.234683 THz : Rb87: F=1 → F'=2

=== Matches in Rb-85 ===

Matched ratio: [1.         0.52500414 1.        ]
  384.229089 THz : Rb85: (F=3→F'=2) × (F=3→F'=3)
  384.229150 THz : Rb85: (F=3→F'=2) × (F=3→F'=4)
  384.229181 THz : Rb85: (F=3→F'=3) × (F=3→F'=4)
  384.229242 THz : Rb85: F=3 → F'=4

Matched ratio: [1.         0.46258589 1.        ]
  384.232079 THz : Rb85: (F=2→F'=1) × (F=2→F'=2)
  384.232110 THz : Rb85: (F=2→F'=1) × (F=2→F'=3)
  384.232125 THz : Rb85: (F=2→F'=2) × (F=2→

In [28]:
data41_T = [0.3753576, 0.3759862, 0.3763094, 0.3769082]
data41_1 = [-0.16011896809768647, -0.6042796851057012, -0.38724909099684934, -0.12374176143111046]
data41_2 = [0.8196752774634963, 0.8138508441244201, 0.8088841149014255, 0.8028361919697905]

data41_T2 = [ 0.3866972,  0.3880268, 0.3888092, 0.3893496, 0.3901556, 0.3909764]
data41_12 = [-0.05048662215351778, -0.2408329309370876, -0.09320802494957436, -0.03726804368028681, -0.02865581124539148, -0.010143825090629614]
data41_22 = [0.6955279375956299, 0.6795106837891742, 0.6715009129710133, 0.665594029744544, 0.6581367734061861, 0.6478125839733568]
exp = np.array(data41_T2)
exp = np.sort(exp)
exp_d = np.diff(exp)              # N-1개 간격
exp_ratio = exp_d / exp_d[0]      # 비율
N = len(exp)                      # 실험 포인트 개수
matches_87 = find_matching_groups(rb87, exp_ratio, N)
matches_85 = find_matching_groups(rb85, exp_ratio, N)
print("=== Matches in Rb-87 ===")
for block, labels, ratio in matches_87:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

print("\n=== Matches in Rb-85 ===")
for block, labels, ratio in matches_85:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

=== Matches in Rb-87 ===

=== Matches in Rb-85 ===


In [37]:
import numpy as np
from itertools import combinations


# ====================================================
# 1) Hyperfine data from Steck (ALL in MHz!!)
# ====================================================

# Rb-87 5P3/2 hyperfine (MHz)
P_87 = {
    0: -302.073800,
    1: -229.851800,
    2:  -72.911200,
    3:  193.740700
}

# Rb-87 5S1/2 hyperfine (MHz)
S_87 = {
    1: -4271.676631815181,
    2: 2563.005979089109
}

# Rb-85 5P3/2 hyperfine (MHz)
P_85 = {
    1: -113.307000,
    2:  -83.955000,
    3:  -20.503000,
    4:  100.357000
}

# Rb-85 5S1/2 hyperfine (MHz)
S_85 = {
    2: -1770.8439228,
    3: 1264.8885163
}


# ====================================================
# 2) Fine-structure COG frequencies (Hz)
#    (Steck D2 line = 384.2304844685 THz)
# ====================================================

ref_87 = 384.2304844685e12   # Hz
ref_85 = 384.2304063730e12   # Hz


# ====================================================
# 3) Build allowed transitions
# ====================================================
def make_allowed(F_list, Fp_list, P, S, ref):
    """
    Steck 공식:
    nu = ref + (P[Fp] - S[F]) * 1e6   (MHz → Hz)
    """
    out = {}
    for F in F_list:
        for Fp in Fp_list:
            if abs(F - Fp) <= 1:
                nu = ref + (P[Fp] - S[F]) * 1e6
                out[(F, Fp)] = nu
    return out


# ====================================================
# 4) Build crossovers (midpoint of same-F allowed)
# ====================================================
def make_crossovers(allowed):
    out = {}
    keys = list(allowed.keys())
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            (F1, Fp1) = keys[i]
            (F2, Fp2) = keys[j]
            if F1 == F2:
                f1 = allowed[(F1, Fp1)]
                f2 = allowed[(F2, Fp2)]
                out[((F1, Fp1), (F2, Fp2))] = 0.5*(f1 + f2)
    return out


# ====================================================
# 5) Pack block: allowed + crossover sorted
# ====================================================
def pack_block(allowed, cross):
    arr = []
    for (F, Fp), f in allowed.items():
        arr.append((f, f"Allowed F={F}→F'={Fp}"))
    for (k1, k2), f in cross.items():
        (F1, Fp1) = k1
        (F2, Fp2) = k2
        arr.append((f, f"CO ({F1}->{Fp1})×({F2}->{Fp2})"))
    return sorted(arr, key=lambda x: x[0])


# ====================================================
# 6) Build four pure blocks
# ====================================================
lines_87_F1 = pack_block(
    make_allowed([1], [0, 1, 2], P_87, S_87, ref_87),
    make_crossovers(make_allowed([1], [0, 1, 2, 3], P_87, S_87, ref_87))
)

lines_87_F2 = pack_block(
    make_allowed([2], [1, 2, 3], P_87, S_87, ref_87),
    make_crossovers(make_allowed([2], [1, 2, 3], P_87, S_87, ref_87))
)

lines_85_F2 = pack_block(
    make_allowed([2], [1, 2, 3], P_85, S_85, ref_85),
    make_crossovers(make_allowed([2], [1, 2, 3], P_85, S_85, ref_85))
)

lines_85_F3 = pack_block(
    make_allowed([3], [2, 3, 4], P_85, S_85, ref_85),
    make_crossovers(make_allowed([3], [2, 3, 4], P_85, S_85, ref_85))
)


# ====================================================
# 7) Matching function (Version B)
# ====================================================
def find_matching_groups(group, exp_ratio, N, tol=0.15):
    matches = []
    freqs = np.array([f for f, _ in group])
    labels = [lab for _, lab in group]

    for idx in combinations(range(len(freqs)), N):
        block = freqs[list(idx)]
        block_label = [labels[i] for i in idx]

        block = np.sort(block)
        d = np.diff(block)
        if d[0] == 0:
            continue

        ratio = d / d[0]
        rel_err = np.abs(ratio - exp_ratio) / ratio

        if np.all(rel_err < tol):
            matches.append((block, block_label, ratio))

    return matches


# ====================================================
# 8) Run matching for data41_T2
# ====================================================

exp = np.sort(data4_T)

exp_d = np.diff(exp)
exp_ratio = exp_d / exp_d[0]   # first-gap normalized
N = len(exp)

# Test four blocks
blocks = {
    "Rb87 F=1 block": lines_87_F1,
    "Rb87 F=2 block": lines_87_F2,
    "Rb85 F=2 block": lines_85_F2,
    "Rb85 F=3 block": lines_85_F3
}

results = {name: find_matching_groups(blk, exp_ratio, N)
           for name, blk in blocks.items()}

print(results)


{'Rb87 F=1 block': [(array([3.84234490e+14, 3.84234569e+14, 3.84234605e+14, 3.84234683e+14]), ['CO (1->0)×(1->1)', 'CO (1->0)×(1->2)', 'CO (1->1)×(1->2)', "Allowed F=1→F'=2"], array([1.        , 0.46018685, 1.        ]))], 'Rb87 F=2 block': [(array([3.84227770e+14, 3.84227903e+14, 3.84227982e+14, 3.84228115e+14]), ['CO (2->1)×(2->2)', 'CO (2->1)×(2->3)', 'CO (2->2)×(2->3)', "Allowed F=2→F'=3"], array([1.        , 0.58855984, 1.        ]))], 'Rb85 F=2 block': [(array([3.84232079e+14, 3.84232110e+14, 3.84232125e+14, 3.84232157e+14]), ['CO (2->1)×(2->2)', 'CO (2->1)×(2->3)', 'CO (2->2)×(2->3)', "Allowed F=2→F'=3"], array([1.        , 0.46258589, 1.        ]))], 'Rb85 F=3 block': [(array([3.84229089e+14, 3.84229150e+14, 3.84229181e+14, 3.84229242e+14]), ['CO (3->2)×(3->3)', 'CO (3->2)×(3->4)', 'CO (3->3)×(3->4)', "Allowed F=3→F'=4"], array([1.        , 0.52500414, 1.        ]))]}


In [43]:
import numpy as np
from itertools import combinations

# ==========================
# 1. Hyperfine energies (MHz)
# ==========================

# Rb-87, 5P3/2
P_87 = {
    0: -302.073800,
    1: -229.851800,
    2:  -72.911200,
    3:  193.740700
}

# Rb-87, 5S1/2
S_87 = {
    1: -4271.676631815181,
    2: 2563.005979089109
}

# Rb-85, 5P3/2
P_85 = {
    1: -113.307000,
    2:  -83.955000,
    3:  -20.503000,
    4:  100.357000
}

# Rb-85, 5S1/2
S_85 = {
    2: -1770.8439228,
    3: 1264.8885163
}

# ================================
# 2. D2 line center-of-gravity (Hz)
# ================================
ref_87 = 384.2304844685e12
ref_85 = 384.2304063730e12

# ==========================
# 3. Allowed transition 만들기
# ==========================

def make_allowed(F_list, allowed_Fp_list, P, S, ref):
    out = {}
    for F in F_list:
        for Fp in allowed_Fp_list:
            nu = ref + (P[Fp] - S[F]) * 1e6  # convert MHz → Hz
            out[(F, Fp)] = nu
    return out

# ==========================
# 4. Crossover 만들기
# ==========================

def make_crossovers(allowed):
    out = {}
    keys = list(allowed.keys())
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            (F1, Fp1) = keys[i]
            (F2, Fp2) = keys[j]
            if F1 == F2:
                f1 = allowed[(F1, Fp1)]
                f2 = allowed[(F2, Fp2)]
                out[((F1, Fp1), (F2, Fp2))] = 0.5*(f1 + f2)
    return out

# ==========================
# 5. Pack block (allowed + crossover)
# ==========================

def pack_block(allowed, cross):
    arr = []
    for (F, Fp), f in allowed.items():
        arr.append((f, f"Allowed F={F}→F'={Fp}"))
    for ((F1, Fp1), (F2, Fp2)), f in cross.items():
        arr.append((f, f"CO ({F1}->{Fp1})×({F2}->{Fp2})"))
    return sorted(arr, key=lambda x: x[0])

# ==========================
# 6. 4개 블록 생성 (F′=0 제거 반영)
# ==========================

lines_87_F1 = pack_block(
    make_allowed([1], [0,1,2], P_87, S_87, ref_87),
    make_crossovers(make_allowed([1], [0,1,2], P_87, S_87, ref_87))
)

lines_87_F2 = pack_block(
    make_allowed([2], [1,2,3], P_87, S_87, ref_87),
    make_crossovers(make_allowed([2], [1,2,3], P_87, S_87, ref_87))
)

lines_85_F2 = pack_block(
    make_allowed([2], [1,2,3], P_85, S_85, ref_85),
    make_crossovers(make_allowed([2], [1,2,3], P_85, S_85, ref_85))
)

lines_85_F3 = pack_block(
    make_allowed([3], [2,3,4], P_85, S_85, ref_85),
    make_crossovers(make_allowed([3], [2,3,4], P_85, S_85, ref_85))
)

# ==========================
# 7. 비율 계산 함수
# ==========================

def print_block_ratios(name, block):
    print(f"\n=== {name} ===")

    # 주파수 기준 정렬
    sorted_block = sorted(block, key=lambda x: x[0])
    freqs  = np.array([f for f,_ in sorted_block])
    labels = [lab for _,lab in sorted_block]

    d = np.diff(freqs)
    ratio = d / d[0]

    print(f"{'From':40s} {'To':40s} {'Δf (Hz)':>15s} {'ratio':>10s}")
    print("-"*110)

    for i in range(len(d)):
        print(f"{labels[i]:40s} {labels[i+1]:40s} {d[i]:15.3f} {ratio[i]:10.6f}")

# ==========================
# 8. 결과 출력
# ==========================
def print_exp_ratio(data):
    print("\n=== Experimental data41_T ===")
    exp = np.sort(np.array(data))
    d = np.diff(exp)
    ratio = d / d[0]
    print("간격:", d)
    print("ratio:", ratio)

print_exp_ratio(data41_T2)




print_block_ratios("Rb87 F=1 block", lines_87_F1)
print_block_ratios("Rb87 F=2 block", lines_87_F2)
print_block_ratios("Rb85 F=2 block", lines_85_F2)
print_block_ratios("Rb85 F=3 block", lines_85_F3)


print_block_ratios("Rb87 F=1 block", lines_87_F1)
print_block_ratios("Rb87 F=2 block", lines_87_F2)
print_block_ratios("Rb85 F=2 block", lines_85_F2)
print_block_ratios("Rb85 F=3 block", lines_85_F3)


=== Experimental data41_T ===
간격: [0.0013296 0.0007824 0.0005404 0.000806  0.0008208]
ratio: [1.         0.58844765 0.40643803 0.60619735 0.61732852]

=== Rb87 F=1 block ===
From                                     To                                               Δf (Hz)      ratio
--------------------------------------------------------------------------------------------------------------
Allowed F=1→F'=0                         CO (1->0)×(1->1)                            36111000.000   1.000000
CO (1->0)×(1->1)                         Allowed F=1→F'=1                            36111000.000   1.000000
Allowed F=1→F'=1                         CO (1->0)×(1->2)                            42359300.000   1.173030
CO (1->0)×(1->2)                         CO (1->1)×(1->2)                            36111000.000   1.000000
CO (1->1)×(1->2)                         Allowed F=1→F'=2                            78470300.000   2.173030

=== Rb87 F=2 block ===
From                                